# Exploratory Data Analysis 5.0

> Goal: Build rolling team-form features, export a rolling-feature dataset, and evaluate an updated baseline logistic regression model.

## 1) Load and Order Match Data

In [18]:
import pandas as pd

matches_df = pd.read_csv('../data/processed/processed_matches.csv')
matches_df['date'] = pd.to_datetime(matches_df['date'])
matches_df = matches_df.sort_values('date').reset_index(drop=True)

matches_df.head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1
4,LOLTMNT05_171066,2026-01-09 17:09:20,16.01,LIT,HMBLE,P11 Esports,1


## 2) Engineer All-Time + Rolling Features

Compute lagged all-time win-rate features and last-5-match rolling features per team before each match is played.

In [19]:
window_size = 5

team_history = {}
team_totals = {}

feature_rows = []

for _, row in matches_df.iterrows():
    blue = row["blue_team"]
    red = row["red_team"]

    # initialize
    if blue not in team_history:
        team_history[blue] = []
        team_totals[blue] = {"wins": 0, "games": 0}

    if red not in team_history:
        team_history[red] = []
        team_totals[red] = {"wins": 0, "games": 0}

    # --- ALL-TIME ---
    blue_games = team_totals[blue]["games"]
    red_games = team_totals[red]["games"]

    blue_wr = team_totals[blue]["wins"] / blue_games if blue_games > 0 else 0.5
    red_wr = team_totals[red]["wins"] / red_games if red_games > 0 else 0.5

    # --- ROLLING ---
    blue_recent = team_history[blue][-window_size:]
    red_recent = team_history[red][-window_size:]

    blue_games_5 = len(blue_recent)
    red_games_5 = len(red_recent)

    blue_wr_5 = sum(blue_recent) / blue_games_5 if blue_games_5 > 0 else 0.5
    red_wr_5 = sum(red_recent) / red_games_5 if red_games_5 > 0 else 0.5

    feature_rows.append({
        **row,

        # ALL-TIME
        "blue_team_wr": blue_wr,
        "red_team_wr": red_wr,
        "blue_team_games": blue_games,
        "red_team_games": red_games,
        "wr_diff": blue_wr - red_wr,

        # ROLLING
        "blue_team_wr_5": blue_wr_5,
        "red_team_wr_5": red_wr_5,
        "blue_team_games_5": blue_games_5,
        "red_team_games_5": red_games_5,
        "wr_diff_5": blue_wr_5 - red_wr_5,
    })

    # --- UPDATE AFTER ---
    if row["blue_side_win"] == 1:
        team_totals[blue]["wins"] += 1
        team_history[blue].append(1)

        team_history[red].append(0)
    else:
        team_totals[red]["wins"] += 1
        team_history[blue].append(0)
        team_history[red].append(1)

    team_totals[blue]["games"] += 1
    team_totals[red]["games"] += 1

combined_df = pd.DataFrame(feature_rows)
combined_df.head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
4,LOLTMNT05_171066,2026-01-09 17:09:20,16.01,LIT,HMBLE,P11 Esports,1,0.0,1.0,1,1,-1.0,0.0,1.0,1,1,-1.0


## 3) Inspect Rolling Features

In [20]:
combined_df[[
    "blue_team_wr_5",
    "red_team_wr_5",
    "blue_team_games_5",
    "red_team_games_5",
    "wr_diff_5"
]].head(10)

,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,0.5,0.5,0,0,0.0
1,0.5,0.5,0,0,0.0
2,0.5,0.5,0,0,0.0
3,0.5,0.5,0,0,0.0
4,0.0,1.0,1,1,-1.0
5,0.0,1.0,1,1,-1.0
6,0.0,1.0,1,1,-1.0
7,1.0,0.0,1,1,1.0
8,0.5,0.5,0,0,0.0
9,1.0,0.0,1,1,1.0


In [21]:
combined_df[["blue_team_wr_5", "red_team_wr_5"]].describe()

,blue_team_wr_5,red_team_wr_5
count,2792.000000,2792.000000
mean,0.528522,0.498520
std,0.289908,0.285902
min,0.000000,0.000000
25%,0.400000,0.333333
50%,0.600000,0.500000
75%,0.800000,0.666667
max,1.000000,1.000000


In [22]:
combined_df[combined_df["blue_team_games_5"] == 0].head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win,blue_team_wr,red_team_wr,blue_team_games,red_team_games,wr_diff,blue_team_wr_5,red_team_wr_5,blue_team_games_5,red_team_games_5,wr_diff_5
0,LOLTMNT05_171038,2026-01-08 17:08:27,16.01,LIT,GMBLERS Esports,EKO Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
1,LOLTMNT05_172024,2026-01-08 17:56:21,16.01,LIT,Deacoy,Zena Esports,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
2,LOLTMNT05_171043,2026-01-08 18:51:06,16.01,LIT,Axolotl,P11 Esports,0,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
3,LOLTMNT05_171051,2026-01-08 19:44:59,16.01,LIT,Colossal Gaming,HMBLE,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0
8,LOLTMNT03_335584,2026-01-12 05:10:10,16.01,LCKC,Nongshim Esports Academy,DN SOOPers Challengers,1,0.5,0.5,0,0,0.0,0.5,0.5,0,0,0.0


## 4) Export Rolling Feature Dataset

Persist the combined feature table for downstream training and evaluation.

In [ ]:
combined_df.to_csv("../data/processed/rolling_feature_matches.csv", index=False)

## 5) Train and Evaluate Baseline Model

Train a logistic regression model using both all-time and rolling form features.

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model_df = combined_df.copy()

features = [
    "wr_diff",
    "blue_team_games",
    "red_team_games"
]

X = model_df[features]
y = model_df["blue_side_win"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.6064400715563506
Classification Report:
               precision    recall  f1-score   support

           0       0.56      0.64      0.60       255
           1       0.66      0.58      0.62       304

    accuracy                           0.61       559
   macro avg       0.61      0.61      0.61       559
weighted avg       0.61      0.61      0.61       559



## 6) Inspect Model Coefficients

Review feature coefficients to understand directional influence in the classifier.

In [25]:
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})

print(coefficients)

             feature  coefficient
0       blue_team_wr     0.882229
1        red_team_wr    -0.146651
2    blue_team_games     0.013154
3     red_team_games    -0.001926
4            wr_diff     1.028880
5     blue_team_wr_5    -0.117266
6      red_team_wr_5    -0.333030
7  blue_team_games_5     0.080271
8   red_team_games_5    -0.085453
9          wr_diff_5     0.215765
